In [361]:
%config InlineBackend.figure_format = 'svg'

```{index} ChemPy
```
```{index} simulations
```
(18)=
# Chapter 18: ChemPy

The ChemPy library provides a collection of tools useful for general, physical, analytical, and inorganic chemistry. Examples include balancing chemical reactions, stoichiometric calculations, solving complex equilibria, and decomposing reducible symmetry representations. This package does not come with Anaconda installations and is not automatically included in [Colab](0.2.2), so you will need to install it using either `pip` or `conda`. See the most recent [package instructions](https://github.com/bjodah/chempy) for further details.

In [362]:
from collections import defaultdict

```{index} SMILES
```
```{index} InChI
```
(18.1)=
## 15.1 Formulas, Stoichiometry, and Equilibria

(18.1.1)=
### Chemical Formulas

Before we move into solving more complex problems, we first need to introduce a the foundational object in ChemPy, which is the `Substance`. The `Substance` holds information about a chemical including the chemical composition, mass, charge, etc. The substance holds information about the chemical substance that is always true. There is a subclass called a `Species`, which inherits attributes from the `Substance` class but also allows phase information used in kinetics and equilibrium calculations. Both are imported as shown below.

In [363]:
from chempy import Substance, Species

The most basic way to create a `Substance` is using the `Substance()` function, which accepts a number of parameters including. These are keyword arguments, but the `name` is frequently just supplied as the first positional argument.

| Parameter | Type | Description |
|:---------:|:------:|:--------------|
| `name` | `str` | Name of substance |
| `composition` | `dict` | Elemental composition |
| `charge` | `int` | Overall net charge | 
| `data` | `dict` |  Dictionary with user-defined keys |

The `Substance` and `Species` objects include the following attributes, among others.

| Attribute | Type | Description |
|:---------:|:------:|:--------------|
| `mass` | `str` | Name of substance |
| `composition` | `dict` | Elemental composition |
| `charge` | `int` | Overall net charge | 
| `data` | `dict` |  Dictionary with user-defined keys |

In the example below, we create an ammonia substance. No parameters are required when a `Substance` is created, but including some attributes is helpful for later applications. Below, we will include the elemental composition of ammonia using a dictionary where the element's atomic number as the keys. For example, ammonia has one nitrogen (AN = 7) and three hydrogens (AN = 1).

In [364]:
NH3 = Substance('NH3', composition={7: 1, 1: 3})

Alternatively, you can create a `Substance` object using the `from_formula` constructor. This function parses a string molecular formula of the chemical substance and uses it to populate the name and composition of the `Substance`.

In [365]:
sulf = Substance.from_formula('Na2SO4')
sulf.name

'Na2SO4'

In [366]:
sulf.composition

{8: 4, 11: 2, 16: 1}

In [367]:
sulf.charge

0

The charge defaults to 0 unless a charge is included in the formula. If the 

In [368]:
nitrate = Substance.from_formula('NO3-')
nitrate.charge

-1

In [369]:
copper = Substance.from_formula('Cu2+2')
copper.charge

2

Once a `Substance` has been created, we can us it to calculate the mass of the substance like below.

In [370]:
sulf.mass

142.03553856000002

In [371]:
sulf.mass

142.03553856000002

In [372]:
benzene = Substance('C6H6', data={'bp': 80})
benzene

<Substance(name=C6H6, ...)>

In [373]:
sulf.charge

0

In [374]:
sulf.latex_name

'Na_{2}SO_{4}'

In [375]:
cisplat = Substance('cisplat', composition={78: 1, 7: 2, 17: 2, 1: 6})

In [376]:
cisplat.mass

300.04600000000005

(18.1.2)=
### Stoichiometry


In [377]:
from chempy import balance_stoichiometry

In [378]:
r, p = balance_stoichiometry({'Cu', 'HNO3'}, {'Cu(NO3)2', 'NO', 'H2O'})
r

OrderedDict([('Cu', 3), ('HNO3', 8)])

In [379]:
p

OrderedDict([('Cu(NO3)2', 3), ('H2O', 4), ('NO', 2)])

In [380]:
balance_stoichiometry({'P4', 'OH-', 'H2O'}, {'PH3', 'H2PO4-'})

(OrderedDict([('H2O', 9), ('OH-', 3), ('P4', 2)]),
 OrderedDict([('H2PO4-', 3), ('PH3', 5)]))

(18.1.3)=
### Calculating Theoretical Yields

The ChemPy library does not include a direct feature to calculate the limiting reactant or theoretical yield, but a short Python script can be written leveraging the above functionality. As our example, we will use the oxidation of nitrogen monooxide to nitrogen dioxide: NO(g) + O$_2$(g) $\rightarrow$ NO$_2$(g).

To carry out this calculation, we will use the following equations where $\xi$(xi) is the extent of reaction, $n_i$ is the initial moles of a chemical species, $n_f$ is the final moles of a chemical species, and $\nu$ is the coefficient for the balanced reaction with positive for products and negative for reactants. ChemPy will contribute to our calculations by balancing the chemical reaction and calculating the molecular weight from the formulas. 
$$ n_f = n_i + \nu\xi $$

In [381]:
grams = [{'NO': 3.01, 'O2': 2.83}, {'NO2': 0.0}]

In [382]:
react, prod = balance_stoichiometry([key for key in grams[0].keys()], 
                                    [key for key in grams[1].keys()])

In [383]:
mol_r = {spec: g / Substance.from_formula(spec).mass for spec, g in grams[0].items()}
xi = min(mol_r[spec] / react[spec] for spec in react.keys())

In [384]:
prod = {spec: xi * mol * Substance.from_formula(spec).mass for spec, mol in prod.items()}
react = {spec: grams[0][spec] - xi * mol * Substance.from_formula(spec).mass for spec, mol in react.items()}

In [385]:
prod

{'NO2': 4.61491201759648}

In [386]:
react

{'NO': 0, 'O2': 1.22508798240352}

We can package this into a Python function which accepts the starting grams of reactant and product in dictionaries.

In [387]:
def calc_grams(grams_r, grams_p):
    """
    """
    react, prod = balance_stoichiometry([key for key in grams_r.keys()], 
                                 [key for key in grams_p.keys()])

    mol_r = {spec: g / Substance.from_formula(spec).mass for spec, g in grams_r.items()}
    xi = min(mol_r[spec] / react[spec] for spec in react.keys())
    prod = {spec: grams_p[spec] + xi * mol * Substance.from_formula(spec).mass 
            for spec, mol in prod.items()}
    react = {spec: grams_r[spec] - xi * mol * Substance.from_formula(spec).mass 
             for spec, mol in react.items()}

    return react, prod

In [388]:
calc_grams({'C6H12O6': 25, 'O2': 40}, {'CO2': 0.0, 'H2O': 0.0})

({'C6H12O6': 0, 'O2': 13.3580896556318},
 {'CO2': 36.6424099114101, 'H2O': 14.9995004329581})

In [389]:
calc_grams({'Mg': 2.40, 'O2': 10.00}, {'MgO': 0.00})

({'Mg': 0, 'O2': 8.42017691832956}, {'MgO': 3.97982308167044})

In [390]:
calc_grams({'TiCl4': 1000, 'Mg': 200}, {'Ti': 0.00, 'MgCl2': 0.00})

({'TiCl4': 219.637934581362, 'Mg': 0},
 {'Ti': 196.943015840362, 'MgCl2': 783.419049578276})


(18.2)=
## 15.2 Kinetic Simulations




(18.3)=
## 15.3 Solving Equilibria



In [391]:
from chempy import Equilibrium
from chempy.equilibria import EqSystem

In [392]:
NH3 = Substance.from_formula('NH3')
N2  = Substance.from_formula('N2')
H2  = Substance.from_formula('H2')

eq = Equilibrium({'NH3': 2}, {'N2': 1, 'H2': 3}, 3.44)
eqsys = EqSystem([eq], [NH3, N2, H2])
iconc = {'NH3': 0.6, 'N2': 0.0, 'H2': 0.6}
x, sol, sane = eqsys.root(iconc)

In [393]:
x

array([0.25998632, 0.17000684, 1.11002052])

In [394]:
ortho = Substance('ortho', composition={6: 9, 1: 12})
meta  = Substance('meta',  composition={6: 9, 1: 12})
para  = Substance('para',  composition={6: 9, 1: 12})

eq_om = Equilibrium({'ortho': 1}, {'meta': 1}, 7.2)
eq_mp = Equilibrium({'para': 1}, {'meta': 1}, 2.47)
eq_po = Equilibrium({'ortho': 1}, {'para': 1}, 2.9)

eqsys = EqSystem([eq_om, eq_mp, eq_po], [ortho, meta, para])
iconc = {'ortho': 1.0 , 'meta': 0.0, 'para': 0.0}

x, sol, sane = eqsys.root(iconc)

In [395]:
x

array([0.09014995, 0.64796585, 0.26188421])

In [396]:
for p, r in ((1, 0), (1, 2), (2, 0)):
    print(x[p] / x[r])

7.1876454796010645
2.4742455718596554
2.9049846795113363


In [397]:
AgBr = Species.from_formula('AgBr(s)')
Ag = Species.from_formula('Ag')
Br = Species.from_formula('Br')
S2O3 = Species.from_formula('S2O3')
AgS2O3 = Species.from_formula('Ag(S2O3)2')

eq1 = Equilibrium({'AgBr(s)': 1}, {'Ag': 1, 'Br': 1}, 5.0e-13)
eq2 = Equilibrium({'Ag': 1, 'S2O3': 2}, {'Ag(S2O3)2': 1}, 4.7e13)

eqsys = EqSystem([eq1, eq2], [AgBr, Ag, Br, S2O3, AgS2O3])

iconc = defaultdict(float, {'AgBr(s)': 1.0, 'S2O3': 1.00})

In [398]:
eq_tot = eq1 + eq2
eq_tot.param

23.5

In [399]:
5.0e-13 * 4.7e13

23.5

In [400]:
Ag = Species.from_formula('Ag+')
Cl = Species.from_formula('Cl-')

eq = Equilibrium({}, {'Ag+': 1, 'Cl-': 1}, 1.8e-10)
eqsys = EqSystem([eq], [Ag, Cl], dont_check={'balance'})

x, sol, sane = eqsys.root({'Ag+': 0, 'Cl-': 0})

In [401]:
x[0]**2

np.float64(1.799999999676002e-10)

## equilibrium activity coefficient example???

(18.3)=
## 18.3 Symmetry

Molecular symmetry is important to chemistry in variety a of applications including bonding, chirality, electronic transitions, and chemical spectroscopy to name a few. While chemists frequently apply symmetry on a qualitative level, group theory can be used as a more rigorous, qualitative treatment of the subject. While the application of group theory to chemistry is a very powerful tool, it often involves an onerous amount matrix math (i.e., linear algebra) when done by hand. These are the kind of calculations that computers are exceptionally well suited at. The `chempy.symmetry` module includes tools specifically to carry out these types of calculations such as decomposing (i.e., reducing) reducible representations, predicting IR- and Raman-active vibrational modes, generating reducible representations for all motions, and generating symmetry-adapted linear combinations (SALCs) of atomic orbitals among others. In this section, we will examine a few of these features for removing some of the tedium from group theory calculations.

The `chempy.symmetry` module has two submoules, `representations` and `salcs`. The `representations` module works with reducible and irreducible representations and vibrational predictions while the `salcs` module predicts SALCs by either the projection operator method or using the symmetry functions in character tables. We will address both here.

(18.3.1)=
### 18.3.1 Symmetry Representations

The first module in the `chempy.symmetry` module is `representations`, which works with interconverting reducible and irreducible representations, generating reducible representations, and predicting vibrational modes as a result of these representations. 

In [402]:
import chempy.symmetry.representations as reps

The main object in the `representations` module is a reducible representation creating using the `Reducible()` function. This requires the values (`gamma`) provided as a list, tuple, or array followed by the point group name (`group`) as a string. The `all_motion=` keyword argument is set to True is the reducible prepresentation is for all motions (i.e., rotation, vibration, and translation) and `False` if it represents vibrations. The default is `False`.

The `gamma` attribute

In [403]:
reduc = reps.Reducible([9, -1, 3, 1], 'c2v', all_motion=True)
reduc.gamma

[9, -1, 3, 1]

In [404]:
reduc.all_motion

True

Use the `decomp()` method to decompose or reduce the reducible representation. The number of each irreducible representation for that point group is returned in the order in the character table.

In [405]:
reduc.decomp()

array([3, 1, 3, 2])

If you don't have a character table available, you can use the `print_table()` function to print character table.

In [406]:
print_table('c2v')

╭─────┬───┬────┬────┬─────╮
│ C2v │ E │ C₂ │ σv │ σvʹ │
├─────┼───┼────┼────┼─────┤
│ A1  │ 1 │ 1  │ 1  │ 1   │
├─────┼───┼────┼────┼─────┤
│ A2  │ 1 │ 1  │ -1 │ -1  │
├─────┼───┼────┼────┼─────┤
│ B1  │ 1 │ -1 │ 1  │ -1  │
├─────┼───┼────┼────┼─────┤
│ B2  │ 1 │ -1 │ -1 │ 1   │
╰─────┴───┴────┴────┴─────╯


Alternatively, if the `decomp()` parameter `to_dict` is set to `True`, the number of reach irreducible representation is returned as a dictionary, which is often more interpretable than the default NumPy array.

In [407]:
reduc.decomp(to_dict=True)

{'A1': 3, 'A2': 1, 'B1': 3, 'B2': 2}

To predict the IR- and Raman-active modes, use the `ir_active()` and `raman_active()` methods, which also accept the optional `to_dict=` argument.

In [408]:
reduc.ir_active(to_dict=True)

{'A1': 2, 'A2': 0, 'B1': 1, 'B2': 0}

In [409]:
reduc.raman_active(to_dict=True)

{'A1': 2, 'A2': 0, 'B1': 1, 'B2': 0}

Likewise, the vibrational modes can be predicted using the `vibe_modes()` function. In this water example, all vibrational modes happen to be both IR- and Raman-active.

In [417]:
reduc.vibe_modes(to_dict=True)

{'A1': 2, 'A2': 0, 'B1': 1, 'B2': 0}

(18.3.2)=
### 18.3.2 Reducible Representations for All Motions

The generation of a reducible representation for all motions is often a tedious and challenging task. This is already the case for molecules that lie nicely on the cartesian axes and have rotational operations that are all multiples of 90$^\circ$, such as the D$_{4h}$ molecule XeF$_4$. This problem is even more challenging when dealing with a molecule such as the C$_{3v}$ molecule PH$_3$ because this problem now involves off-axis operations that require trigonometry to solve. The `Reducible` object includes the `from_atom()` constructor that does this math for the user based on the number of atoms that remain stationary during each symmetry operation. 

```{Note}
If you're curious how this calculation is done, each atoms contribution, $R$, is calculated by teh following equation where $x$ is +1 for E and proper rotations and -1 for all other operations and $n$ and $k$ are the order and exponental of the N$^k_n$ symmetry operation. For example, C$_4$ is $x$ = 1, $n$ = 4, and $k$ = 1. Then the number of atoms is multiplied by the per atom contribution.

$$ R = x + 2 \, cos\left(\frac{2\pi k}{n}\right) $$
```

As an example, we will use the C$_{4v}$ molecule XeOF$_4$. We first need to know what operations are part of this point group and their order. For this, the user can use the  `print_header()` function that accepts the point group name as a string and prints out the symmetry operations along the top row of the character table.

In [410]:
print_header('c4v')

E 2C₄ C₂ 2σv 2σd


For this point group, we have E, C$_4$, C$_2$, $\sigma_v$, and $\sigma_d$. Below are the numbers of atoms in the XeOF$_4$ that do **not** move for each of these operations. Remember that $\sigma_v$ runs through the outer atoms while $\sigma_d$ runs between them.

| C$_{4v}$ | E | 2 C$_4$ | C$_2$ | 2 $\sigma_v$ | 2 $\sigma_d$|
|:-------:|:--:|:------:|:------:|:------------:|:-----------:|
|         | 6  | 2      | 2      | 4            |  2          |

By feeding these values as a list or array to the `from_atoms()` method along with the point group, we get the reducible representation for all moptions.

In [411]:
c4v_rep = Reducible.from_atoms([6, 2, 2, 4, 2], 'c4v')
c4v_rep.gamma

array([18,  2, -2,  4,  2])

This method automatically sets `all_motion=True` because it includes rotational, translational, and vibrational modes.

In [412]:
c4v_rep.all_motion

True

With this reducible representation, we can then easily determine the number of vibrational modes (`vibe_modes()`), IR-active modes (`ir_active()`), and Raman-active modes (`raman_active()`).

In [413]:
c4v_rep.vibe_modes(to_dict=True)

{'A1': 3, 'A2': 0, 'B1': 2, 'B2': 1, 'E': 3}

In [414]:
c4v_rep.ir_active(to_dict=True)

{'A1': 3, 'A2': 0, 'B1': 0, 'B2': 0, 'E': 3}

In [415]:
c4v_rep.raman_active(to_dict=True)

{'A1': 3, 'A2': 0, 'B1': 2, 'B2': 1, 'E': 3}

(18.3.3)=
### 18.3.3 Symmetry Adapted Linear Combinations

The `chempy.symmetry.salcs` submodule predicts SALCs by either the projection operator method or using the symmetry functions in character tables

In [416]:
import chempy.symmetry.salcs as salcs

(fr18)=
## Further Reading

1. RDKit

## Exercises

Complete the following exercises in a Jupyter notebook using RDKit. You are encouraged to also use data libraries such as NumPy or pandas to support your solutions. Any data file(s) referred to in the problems can be found in the [data](https://github.com/weisscharlesj/SciCompforChemists/tree/master/notebooks) folder in the same directory as this chapter's Jupyter notebook. Alternatively, you can download a zip file of the data for this chapter from [here](https://github.com/weisscharlesj/data_SciCompforChem) by selecting the appropriate chapter file and then clicking the **Download** button.